In [1]:
# ── Imports ──────────────────────────────────────────
import pandas as pd        # for working with data tables
import numpy as np         # for numbers and math operations
import matplotlib.pyplot as plt  # for basic plots
import seaborn as sns      # for nicer looking plots
import warnings
warnings.filterwarnings('ignore')  # hide unnecessary warnings

print("✅ All libraries loaded successfully!")


✅ All libraries loaded successfully!


In [2]:
# Load the CSV file into a dataframe
df = pd.read_csv('../data/raw/Spotify_Youtube.csv')

print(f"✅ Dataset loaded!")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

✅ Dataset loaded!
Rows: 20718, Columns: 28


In [3]:
# See the first 5 rows
df.head()

,Unnamed: 0,Artist,Url_spotify,Track,Album,Album_type,Uri,Danceability,Energy,Key,...,Url_youtube,Title,Channel,Views,Likes,Comments,Description,Licensed,official_video,Stream
0,0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,https://www.youtube.com/watch?v=HyHNuVaZJ-k,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09
1,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,https://www.youtube.com/watch?v=yYDmaexVHic,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08
2,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,https://www.youtube.com/watch?v=qJa-VFwPpYA,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07
3,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,https://www.youtube.com/watch?v=04mfKJWDSzI,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08
4,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,https://www.youtube.com/watch?v=1V_xRb0x9aw,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08


In [4]:
# See column names, data types, and null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20718 entries, 0 to 20717
Data columns (total 28 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        20718 non-null  int64  
 1   Artist            20718 non-null  object 
 2   Url_spotify       20718 non-null  object 
 3   Track             20718 non-null  object 
 4   Album             20718 non-null  object 
 5   Album_type        20718 non-null  object 
 6   Uri               20718 non-null  object 
 7   Danceability      20716 non-null  float64
 8   Energy            20716 non-null  float64
 9   Key               20716 non-null  float64
 10  Loudness          20716 non-null  float64
 11  Speechiness       20716 non-null  float64
 12  Acousticness      20716 non-null  float64
 13  Instrumentalness  20716 non-null  float64
 14  Liveness          20716 non-null  float64
 15  Valence           20716 non-null  float64
 16  Tempo             20716 non-null  float6

In [5]:
# Count how many nulls are in each column
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent.round(2)
})

# Show only columns that actually have missing values
print(missing_df[missing_df['Missing Count'] > 0])

                  Missing Count  Missing %
Danceability                  2       0.01
Energy                        2       0.01
Key                           2       0.01
Loudness                      2       0.01
Speechiness                   2       0.01
Acousticness                  2       0.01
Instrumentalness              2       0.01
Liveness                      2       0.01
Valence                       2       0.01
Tempo                         2       0.01
Duration_ms                   2       0.01
Url_youtube                 470       2.27
Title                       470       2.27
Channel                     470       2.27
Views                       470       2.27
Likes                       541       2.61
Comments                    569       2.75
Description                 876       4.23
Licensed                    470       2.27
official_video              470       2.27
Stream                      576       2.78


In [6]:
# Stream and Views are our most important columns
# Any row without them is useless — drop them
df.dropna(subset=['Stream', 'Views', 'Likes'], inplace=True)

print(f"✅ Rows after dropping critical nulls: {df.shape[0]}")

✅ Rows after dropping critical nulls: 19625


In [7]:
# Fill missing numeric columns with median (safer than mean)
numeric_cols = ['Danceability','Energy','Key','Loudness','Speechiness',
                'Acousticness','Instrumentalness','Liveness','Valence',
                'Tempo','Duration_ms','Comments']

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

print(f"✅ Numeric nulls filled with median")

# Fill missing text columns with 'Unknown'
df['Description'].fillna('Unknown', inplace=True)
df['Licensed'].fillna('Unknown', inplace=True)

print(f"✅ Text nulls filled with Unknown")

✅ Numeric nulls filled with median
✅ Text nulls filled with Unknown


In [8]:
# Remove rows where same Track by same Artist appears more than once
before = df.shape[0]
df.drop_duplicates(subset=['Artist', 'Track'], inplace=True)
after = df.shape[0]

print(f"✅ Duplicates removed: {before - after} rows dropped")
print(f"Clean dataset size: {df.shape[0]} rows")

✅ Duplicates removed: 77 rows dropped
Clean dataset size: 19548 rows


In [9]:
# Make sure numeric columns are actually numeric
df['Stream'] = pd.to_numeric(df['Stream'], errors='coerce')
df['Views']  = pd.to_numeric(df['Views'],  errors='coerce')
df['Likes']  = pd.to_numeric(df['Likes'],  errors='coerce')

# Convert official_video to boolean
df['official_video'] = df['official_video'].astype(str).str.strip().str.lower()
df['official_video'] = df['official_video'].map({'true': True, 'false': False, '1': True, '0': False})

print("✅ Data types fixed!")
print(df[['Stream','Views','Likes','official_video']].dtypes)

✅ Data types fixed!
Stream            float64
Views             float64
Likes             float64
official_video       bool
dtype: object


In [10]:
# Save to processed folder so other notebooks can use it
df.to_csv('../data/processed/cleaned.csv', index=False)

print(f"✅ Cleaned file saved to data/processed/cleaned.csv")
print(f"Final shape: {df.shape[0]} rows × {df.shape[1]} columns")

✅ Cleaned file saved to data/processed/cleaned.csv
Final shape: 19548 rows × 28 columns
